# 05 · Deep Learning Experiments

Scaffold for transformer-based models (CodeBERT) that jointly represent PR text and code (**RQ1**). Heavy steps are **gated** behind `RUN_HEAVY` so the notebook always runs; a lightweight embedding path runs by default.

- **Inputs:** `configs/codebert.yaml`, `data/sample/sample_prs.csv`
- **Outputs:** A loaded transformer config, a tokenizer demo (gated), and a runnable embedding→classifier illustration.

> ⚠️ **Sample vs. real data.** This notebook runs on the committed 10-row synthetic sample so the toolchain works without PRismBench. The sample has singleton classes, so metrics here are *illustrative only*. Each `TODO` marks where the real dataset in `data/raw/` plugs in.

In [ ]:
# --- Standard setup: locate project root, add src/ to path, load helpers ---
import sys
from pathlib import Path

import pandas as pd


def find_project_root(start: Path) -> Path:
    """Walk upwards until we find the repo root (has pyproject.toml + src/pr_risk)."""
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists() and (p / "src" / "pr_risk").exists():
            return p
    return start


PROJECT_ROOT = find_project_root(Path.cwd())
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

pd.set_option("display.max_columns", 50)
SAMPLE_CSV = PROJECT_ROOT / "data" / "sample" / "sample_prs.csv"
print("Project root :", PROJECT_ROOT)
print("Sample CSV   :", SAMPLE_CSV.name, "| exists:", SAMPLE_CSV.exists())

## 0. Heavy-run switch
Keep `False` for CI / laptops without a GPU. Set `True` only with `torch`+`transformers` installed.

In [ ]:
RUN_HEAVY = False  # TODO: set True on Colab/AWS with a GPU to fine-tune CodeBERT
print("RUN_HEAVY =", RUN_HEAVY)

## 1. Load the transformer config

In [ ]:
from pr_risk.utils.config import load_config

codebert_cfg = load_config(PROJECT_ROOT / "configs" / "codebert.yaml")
codebert_cfg

## 2. Tokenizer demo (gated)
Needs `transformers`. Skipped by default; shows how PR text becomes model input.

In [ ]:
try:
    from transformers import AutoTokenizer
    HAS_TRANSFORMERS = True
except ImportError:
    HAS_TRANSFORMERS = False
print("transformers available:", HAS_TRANSFORMERS)

if RUN_HEAVY and HAS_TRANSFORMERS:
    tok = AutoTokenizer.from_pretrained(codebert_cfg["base_model"])
    enc = tok("Fix null token validation in auth middleware",
              truncation=True, max_length=codebert_cfg["params"]["max_length"])
    print("input_ids[:12]:", enc["input_ids"][:12])
else:
    print("Skipped tokenizer demo (set RUN_HEAVY=True and install transformers to run).")

## 3. Lightweight embedding alternative (runs by default)
Until CodeBERT is wired in, we illustrate the **embedding → classifier** pattern with
TF-IDF vectors as a stand-in for contextual embeddings.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

from pr_risk.data.load_data import load_csv
from pr_risk.features.text_features import create_text_features_tfidf

df = load_csv(SAMPLE_CSV)
df = df[df["is_risky"].isin([0, 1])].copy()
df["text"] = (df["title"].fillna("") + " " + df["code_diff_summary"].fillna("")).str.lower()

train_df, test_df = train_test_split(df, test_size=0.3, random_state=42)
Xtr, _, Xte, _ = create_text_features_tfidf(
    train_df["text"], train_df["text"], test_df["text"], max_features=200
)
clf = LogisticRegression(max_iter=1000).fit(Xtr, train_df["is_risky"])
preds = clf.predict(Xte)
print("TF-IDF + LogReg F1 (placeholder for CodeBERT embeddings):",
      round(f1_score(test_df["is_risky"], preds, average="weighted", zero_division=0), 3))

## 4. Transformer fine-tune scaffold (gated)
The real training loop lives here; implement in `pr_risk.models.train_transformer`.

In [ ]:
if RUN_HEAVY and HAS_TRANSFORMERS:
    # TODO: implement in pr_risk.models.train_transformer
    #   1. Build a HuggingFace Dataset from PR text (+ code diff) and risk_type labels.
    #   2. AutoModelForSequenceClassification.from_pretrained(base_model, num_labels=...)
    #   3. Trainer(...).train() using params in codebert.yaml (batch, epochs, lr).
    #   4. Evaluate with pr_risk.utils.metrics and save to models/transformer/.
    raise NotImplementedError("Fine-tuning is a planned extension.")
else:
    print("Fine-tune scaffold is gated. Steps it will perform:")
    for step in ["tokenize PR text+code", "load CodeBERT seq-classifier",
                 "Trainer.train()", "evaluate + save checkpoint"]:
        print("  -", step)

## Next steps / TODO (real data + GPU)
- Implement `pr_risk.models.train_transformer` (CodeBERT) for multi-class `risk_type`.
- Replace TF-IDF with CodeBERT/GraphCodeBERT embeddings (see `docs/README.md` §6).
- Run on Colab/AWS GPU (`colab/colab_setup_template.ipynb`); mind CodeBERT's 512-token limit.
- Compare against the baselines from **03** and explain with **06**.